## RAG PIPELINEs -Data ingestion to Vector DB pipeline

In [56]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [57]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: AdvanceddatabasePractical.pdf
  ✓ Loaded 26 pages

Processing: rolesOfSE.pdf
  ✓ Loaded 9 pages

Total documents loaded: 35


In [58]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Advanced data base Practical', 'source': '../data/pdf/AdvanceddatabasePractical.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'AdvanceddatabasePractical.pdf', 'file_type': 'pdf'}, page_content='TRIBHUVAN  UNIVERSITY  INSTITUTE  OF  SCIENCE  AND  TECHNOLOGY  SOCH  COLLEGE  OF  IT   LAB  REPORT  OF  “ADVANCE  DATABASE”   \n \n \nSUBMITTED  TO:                                                            SUBMITTED  BY  \nSoch  College  of  IT                                                         Nabin  Magar  (16846)  Ranipauwa,  pokhara                                                     Semester:  8\n t'),
 Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Advanced data base Practical', 'source': '../data/pdf/AdvanceddatabasePractical.pdf', 'total_pages': 26, 'page': 1, 'page_l

In [59]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [60]:
chunks=split_documents(all_pdf_documents)
chunks

Split 35 documents into 45 chunks

Example chunk:
Content: TRIBHUVAN  UNIVERSITY  INSTITUTE  OF  SCIENCE  AND  TECHNOLOGY  SOCH  COLLEGE  OF  IT   LAB  REPORT  OF  “ADVANCE  DATABASE”   
 
 
SUBMITTED  TO:                                                      ...
Metadata: {'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Advanced data base Practical', 'source': '../data/pdf/AdvanceddatabasePractical.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'AdvanceddatabasePractical.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Advanced data base Practical', 'source': '../data/pdf/AdvanceddatabasePractical.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'AdvanceddatabasePractical.pdf', 'file_type': 'pdf'}, page_content='TRIBHUVAN  UNIVERSITY  INSTITUTE  OF  SCIENCE  AND  TECHNOLOGY  SOCH  COLLEGE  OF  IT   LAB  REPORT  OF  “ADVANCE  DATABASE”   \n \n \nSUBMITTED  TO:                                                            SUBMITTED  BY  \nSoch  College  of  IT                                                         Nabin  Magar  (16846)  Ranipauwa,  pokhara                                                     Semester:  8\n t'),
 Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Advanced data base Practical', 'source': '../data/pdf/AdvanceddatabasePractical.pdf', 'total_pages': 26, 'page': 1, 'page_l

## Next part

### Embdding And VectorStoreDB